# Jones 波片与 PBS 投影测量

本例使用 `qtools.calc.waveplate` 的 NumPy 接口。偏振态是 `(H, V)` 基底下的 Jones 向量，波片是 2×2 Jones 矩阵，矩阵按光传播顺序左乘。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qtools.calc.waveplate import (
    apply, bloch, hwp, pbs, probability, qwp, retarder,
    solve_analyzer_angles,
)

H = np.array([1, 0], dtype=complex)
V = np.array([0, 1], dtype=complex)
D = np.array([1, 1], dtype=complex) / np.sqrt(2)
R = np.array([1, 1j], dtype=complex) / np.sqrt(2)


## HWP + PBS-H：线偏振投影

HWP 角度为目标线偏振角的一半。`D` 经过 22.5° HWP 后变成 H，因此 PBS-H 的透射概率为 1。

In [ ]:
d_after_hwp = hwp(22.5) @ D
print('HWP output:', np.round(d_after_hwp, 8))
print('P(D) with HWP + PBS-H:', probability(d_after_hwp, pbs('H')))

angles = np.linspace(0, 90, 181)  # analyzer linear-polarization angle
p_h = probability(apply(D, [hwp(angles / 2)]), pbs('H'))
plt.plot(angles, p_h)
plt.xlabel('input linear polarization angle (deg)')
plt.ylabel('PBS-H probability')
plt.grid(True, alpha=0.3)
plt.show()

## QWP + HWP + PBS-H：任意纯偏振态投影

`solve_analyzer_angles` 返回传播顺序中的波片角度。这里目标是右旋圆偏振 `R`。

In [ ]:
solution_qh = solve_analyzer_angles(R, ('qwp', 'hwp'))
solution_hq = solve_analyzer_angles(R, ('hwp', 'qwp'))
print('QWP -> HWP -> PBS-H:', solution_qh)
print('HWP -> QWP -> PBS-H:', solution_hq)

qh_output = apply(R, [qwp(solution_qh['angles_deg'][0]), hwp(solution_qh['angles_deg'][1])])
hq_output = apply(R, [hwp(solution_hq['angles_deg'][0]), qwp(solution_hq['angles_deg'][1])])
print('QWP/HWP transmission:', probability(qh_output, pbs('H')))
print('HWP/QWP transmission:', probability(hq_output, pbs('H')))

## 任意 retardance 与批量计算

波片角度可以是数组，返回矩阵和输出态会带有同样的 batch 维度。

In [ ]:
angles = np.linspace(0, 180, 361)
measured_retardance = 157.0
outputs = apply(H, [retarder(angles, measured_retardance)])
p_d = probability(outputs, pbs('D'))
plt.plot(angles, p_d)
plt.xlabel('fast-axis angle (deg)')
plt.ylabel('PBS-D probability')
plt.title(f'Retardance = {measured_retardance} deg')
plt.grid(True, alpha=0.3)
plt.show()

## Bloch/Poincare 球追踪

`return_history=True` 返回输入态和每一级波片后的态。下面让 QWP 角度扫描，观察 QWP 后和 HWP 后的轨迹。

In [ ]:
scan = np.linspace(0, 180, 181)
history = apply(H, [qwp(scan), hwp(22.5)], return_history=True)
trajectory = bloch(history)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
u = np.linspace(0, 2 * np.pi, 40)
v = np.linspace(0, np.pi, 20)
ax.plot_wireframe(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)), color='0.8', linewidth=0.4)
ax.plot(trajectory[1, :, 0], trajectory[1, :, 1], trajectory[1, :, 2], label='after QWP')
ax.plot(trajectory[2, :, 0], trajectory[2, :, 1], trajectory[2, :, 2], label='after HWP')
ax.set_xlabel('S1')
ax.set_ylabel('S2')
ax.set_zlabel('S3')
ax.legend()
plt.show()